# 02 — Primeira análise visual da série

Agora que a base está organizada no tempo, vamos visualizar a demanda de bicicletas.

Neste notebook, vamos observar:

- a demanda ao longo do tempo;
- a diferença entre visão horária e diária;
- padrões por hora do dia;
- padrões por dia da semana.

A ideia ainda não é modelar, mas começar a entender o comportamento da série.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np

if Path.cwd().name == "notebooks":
    raiz_projeto = Path.cwd().parent
else:
    raiz_projeto = Path.cwd()

sys.path.append(str(raiz_projeto / "src"))

from visual_utils import (
    CORES,
    grafico_linha_padrao,
    grafico_barra_padrao
)

caminho_base = raiz_projeto / "outputs" / "tabelas" / "base_bike_organizada.csv"
caminho_saida_serie_diaria = raiz_projeto / "outputs" / "tabelas" / "serie_diaria_bike.csv"

print("Base organizada:")
print(caminho_base)

In [ ]:
df = pd.read_csv(caminho_base)

df["data_hora"] = pd.to_datetime(df["data_hora"])

print(f"Linhas: {df.shape[0]}")
print(f"Colunas: {df.shape[1]}")

df.head()

## A série horária

A base está em frequência horária.

Isso significa que cada linha representa a demanda de bicicletas em uma hora específica.

In [ ]:
fig = grafico_linha_padrao(
    df=df,
    x="data_hora",
    y="demanda",
    titulo="Demanda horária de bicicletas",
    labels={
        "data_hora": "Data e hora",
        "demanda": "Demanda"
    },
    altura=500
)

fig.show()

## Agregando a série por dia

A série horária pode ser muito ruidosa.

Para enxergar melhor o comportamento geral, vamos somar a demanda por dia.

In [ ]:
serie_diaria = (
    df.assign(data=df["data_hora"].dt.floor("D"))
    .groupby("data", as_index=False)
    .agg(demanda=("demanda", "sum"))
    .rename(columns={"data": "data_hora"})
)

print(f"Quantidade de dias observados: {serie_diaria.shape[0]}")
print(f"Dias com demanda igual a zero: {(serie_diaria['demanda'] == 0).sum()}")

serie_diaria.head()

In [ ]:
fig = grafico_linha_padrao(
    df=serie_diaria,
    x="data_hora",
    y="demanda",
    titulo="Demanda diária de bicicletas",
    labels={
        "data_hora": "Data",
        "demanda": "Demanda diária"
    },
    altura=500
)

fig.show()

## Padrão por hora do dia

Agora vamos investigar se a demanda muda dependendo da hora.

Esse tipo de análise ajuda a identificar ciclos dentro do próprio dia.

In [ ]:
demanda_por_hora = (
    df.groupby("hora", as_index=False)
    .agg(demanda_media=("demanda", "mean"))
)

fig = grafico_barra_padrao(
    df=demanda_por_hora,
    x="hora",
    y="demanda_media",
    titulo="Demanda média por hora do dia",
    labels={
        "hora": "Hora do dia",
        "demanda_media": "Demanda média"
    },
    altura=450
)

fig.show()

## Padrão por dia da semana

Também podemos verificar se a demanda muda de acordo com o dia da semana.

Esse é um primeiro passo para reconhecer possíveis padrões sazonais.

In [ ]:
ordem_dias = [
    "segunda-feira",
    "terça-feira",
    "quarta-feira",
    "quinta-feira",
    "sexta-feira",
    "sábado",
    "domingo"
]

demanda_por_dia_semana = (
    df.groupby(["dia_semana", "nome_dia_semana"], as_index=False)
    .agg(demanda_media=("demanda", "mean"))
    .sort_values("dia_semana")
)

fig = grafico_barra_padrao(
    df=demanda_por_dia_semana,
    x="nome_dia_semana",
    y="demanda_media",
    titulo="Demanda média por dia da semana",
    labels={
        "nome_dia_semana": "Dia da semana",
        "demanda_media": "Demanda média"
    },
    altura=450
)

fig.update_xaxes(categoryorder="array", categoryarray=ordem_dias)

fig.show()

## Padrão por mês

Por fim, vamos observar a demanda média por mês.

Esse gráfico ajuda a levantar hipóteses sobre variações ao longo do ano.

In [ ]:
demanda_por_mes = (
    df.groupby("mes", as_index=False)
    .agg(demanda_media=("demanda", "mean"))
)

fig = grafico_barra_padrao(
    df=demanda_por_mes,
    x="mes",
    y="demanda_media",
    titulo="Demanda média por mês",
    labels={
        "mes": "Mês",
        "demanda_media": "Demanda média"
    },
    altura=450
)

fig.show()

In [ ]:
serie_diaria.to_csv(
    caminho_saida_serie_diaria,
    index=False,
    encoding="utf-8-sig"
)

print("Série diária salva em:")
print(caminho_saida_serie_diaria)

## Próximo passo

A análise visual mostrou que a demanda varia ao longo do tempo e apresenta padrões por hora, dia da semana e mês.

No próximo notebook, vamos começar a separar melhor esses comportamentos usando componentes da série, média móvel e estacionariedade.